In [11]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde


ROOT = Path("<LOCAL_COINGECKO_ROOT>")

SOURCE_DIRS = [
    ROOT / "coin_gecko",
    ROOT / "coin_gecko_20250910_112201",
    ROOT / "20260531_130117_stablecoins",
]

OUT_DIR = Path("<LOCAL_OUTPUT_DIR>")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COIN_ID = "tether"
COIN_LABEL = "USDT_USD"

PEG_VALUE = 1.0

START_DATE = None
END_DATE = "2026-05-31"

In [12]:
frames = []

for folder in SOURCE_DIRS:
    file_path = folder / f"{COIN_ID}.csv"

    if file_path.exists():
        temp = pd.read_csv(file_path)

        temp["date"] = pd.to_datetime(temp["date"], errors="coerce")
        temp["prices"] = pd.to_numeric(temp["prices"], errors="coerce")

        temp = temp[["date", "prices"]]
        temp = temp.dropna(subset=["date", "prices"])

        frames.append(temp)

df = pd.concat(frames, ignore_index=True)

df = df.sort_values("date")
df = df.drop_duplicates(subset="date", keep="last")

if START_DATE is not None:
    df = df[df["date"] >= START_DATE]

if END_DATE is not None:
    df = df[df["date"] <= END_DATE]

df = df.copy()

print(df["date"].min(), df["date"].max())

2015-03-06 00:00:00 2026-05-31 00:00:00


In [13]:
df["deviation"] = df["prices"] - PEG_VALUE

deviation = df["deviation"].dropna()

In [16]:
import numpy as np

bins = 60

bin_edges = np.linspace(
    deviation.min(),
    deviation.max(),
    bins + 1
)

bin_width = bin_edges[1] - bin_edges[0]

kde = gaussian_kde(deviation)

x_grid = np.linspace(
    deviation.min(),
    deviation.max(),
    400
)

density_scaled = kde(x_grid) * len(deviation) * bin_width